In [1]:
%%writefile cuda_operations.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

// ---------------- VECTOR ADDITION KERNEL ----------------
__global__ void vectorAdd(int *A, int *B, int *C, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if(i < n)
    {
        C[i] = A[i] + B[i];
    }
}

// ---------------- MATRIX MULTIPLICATION KERNEL ----------------
__global__ void matrixMul(int *A, int *B, int *C, int rowsA, int colsA, int colsB)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if(row < rowsA && col < colsB)
    {
        int sum = 0;

        for(int k = 0; k < colsA; k++)
        {
            sum += A[row * colsA + k] * B[k * colsB + col];
        }

        C[row * colsB + col] = sum;
    }
}

int main()
{
    // =========================================================
    // VECTOR ADDITION
    // =========================================================

    const int n = 10;

    int h_A[n] = {1,2,3,4,5,6,7,8,9,10};
    int h_B[n] = {10,9,8,7,6,5,4,3,2,1};

    int h_C[n];

    int *d_A, *d_B, *d_C;

    int size = n * sizeof(int);

    // Allocate GPU memory
    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    // Copy data to GPU
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // Launch kernel
    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    vectorAdd<<<blocks, threads>>>(d_A, d_B, d_C, n);

    // Copy result back
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    cout << "Vector Addition Result:\n";

    for(int i = 0; i < n; i++)
    {
        cout << h_C[i] << " ";
    }

    cout << "\n\n";

    // Free vector memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    // =========================================================
    // MATRIX MULTIPLICATION
    // =========================================================

    int h_M1[4] = {
        1, 2,
        3, 4
    };

    int h_M2[4] = {
        5, 6,
        7, 8
    };

    int h_Result[4];

    int *d_M1, *d_M2, *d_Result;

    int matrixSize = 4 * sizeof(int);

    // Allocate GPU memory
    cudaMalloc((void**)&d_M1, matrixSize);
    cudaMalloc((void**)&d_M2, matrixSize);
    cudaMalloc((void**)&d_Result, matrixSize);

    // Copy matrices to GPU
    cudaMemcpy(d_M1, h_M1, matrixSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_M2, h_M2, matrixSize, cudaMemcpyHostToDevice);

    // Thread configuration
    dim3 threadsPerBlock(2, 2);
    dim3 blocksPerGrid(1, 1);

    // Launch kernel
    matrixMul<<<blocksPerGrid, threadsPerBlock>>>(
        d_M1,
        d_M2,
        d_Result,
        2,
        2,
        2
    );

    // Copy result back
    cudaMemcpy(h_Result, d_Result, matrixSize, cudaMemcpyDeviceToHost);

    cout << "Matrix Multiplication Result:\n";

    for(int i = 0; i < 4; i++)
    {
        cout << h_Result[i] << " ";

        if((i + 1) % 2 == 0)
        {
            cout << endl;
        }
    }

    // Free GPU memory
    cudaFree(d_M1);
    cudaFree(d_M2);
    cudaFree(d_Result);

    return 0;
}

Writing cuda_operations.cu


In [2]:
!nvcc cuda_operations.cu -o cuda_operations

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [3]:
!./cuda_operations

Vector Addition Result:
11 11 11 11 11 11 11 11 11 11 

Matrix Multiplication Result:
19 22 
43 50 
